# 08 — U-Net ile Arazi Örtüsü Sınıflandırması (Sentinel-2, 10 m)

**Amaç:** GeoClimate dokümanındaki derin öğrenme adımı. **U-Net**, görüntüyü parça parça okuyup her piksele sınıf veren bir evrişimli sinir ağıdır
(segmentasyon). 03'teki modeller her pikseli **tek başına** sınıflandırırdı; U-Net pikselin **komşularına da bakar** (tarla şekli, kent dokusu, kıyı çizgisi).

**Akış:**
1. 07'nin GeoTIFF'lerini yükle; her kutuyu kuzeyden güneye **eğitim / doğrulama / test** şeritlerine böl (aralarında 320 m tampon).
2. U-Net (ResNet-34 kodlayıcı, ImageNet ön eğitimli, 6 kanala uyarlanmış) eğit; karma hassasiyet (AMP), erken durdurma, her epoch'ta Drive'a kayıt.
3. Test şeritlerinde değerlendir: Accuracy, Precision, Recall, F1, **IoU (mIoU)**.
4. **Önce RF, sonra U-Net** (GeoClimate): aynı test pikselleri üzerinde Random Forest ile karşılaştır.
5. **H6 — il dışı test (LOCO):** bir il hiç görülmeden eğitilip o ilde test edilir.
6. **Değişim:** model ilk yıl ve 2025 görüntülerine uygulanır → geçiş matrisi (km²) ve değişim haritaları.

**Çalışma zamanı: GPU zorunlu.** *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU.* Süre: ana eğitim ≈ 20–40 dk, LOCO ≈ 30–60 dk.
Oturum koparsa **Tümünü çalıştır** deyin: eğitim kaldığı epoch'tan, LOCO kaldığı ilden devam eder.

## Kurulum: kod GitHub'dan, sonuçlar Drive'a
- **Kod** (`config.py`, `src/`) GitHub reposundan Colab'ın geçici diskine (`/content/yapay-zeka-proje`) çekilir. Her çalıştırmada en güncel sürüm alınır (`git pull`).
- **Veri ve sonuçlar** (`data/`, `results/`) Google Drive'da `Drive'ım/yapay-zeka-proje/` altına yazılır. Colab kapansa da kaybolmaz.
- Repo herkese açık olduğu için **token gerekmez**. Colab'ın **Gizli anahtarlar (Secrets)** bölümünde `GITHUB_TOKEN` tanımlıysa yine kullanılır (repo gizli yapılırsa gerekir); token ekrana yazdırılmaz.

In [ ]:
!pip install -q segmentation-models-pytorch rasterio

In [ ]:
import os, sys, subprocess, importlib

GITHUB_REPO = "selinzeykaya/yapay-zeka-proje"
GITHUB_BRANCH = "main"   # kendi dalınızdaki kodu çalıştırmak için dal adını yazın (ör. "amine/geoclimate-uyum")
CODE_DIR = "/content/yapay-zeka-proje"                 # kod (GitHub'dan)
WORK_DIR = "/content/drive/MyDrive/yapay-zeka-proje"   # veri + sonuçlar (Drive)

def _git(args, token):
    r = subprocess.run(["git"] + args, capture_output=True, text=True)
    if r.returncode != 0:
        mesaj = r.stderr or r.stdout
        raise RuntimeError("git hatası: " + (mesaj.replace(token, "***") if token else mesaj))

try:
    from google.colab import drive, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    # Repo herkese açık: token gerekmez. Colab Secrets'ta GITHUB_TOKEN varsa kullanılır (repo gizli yapılırsa gerekir).
    try:
        token = userdata.get("GITHUB_TOKEN") or ""
    except Exception:
        token = ""
    adres = f"https://{token}@github.com/{GITHUB_REPO}.git" if token else f"https://github.com/{GITHUB_REPO}.git"
    if not os.path.exists(os.path.join(CODE_DIR, ".git")):
        _git(["clone", "-q", adres, CODE_DIR], token)
    # Seçilen dalın en güncel hâline geç (Colab kopyasındaki yerel değişiklikler atılır)
    _git(["-C", CODE_DIR, "fetch", "-q", "origin", GITHUB_BRANCH], token)
    _git(["-C", CODE_DIR, "checkout", "-q", "-B", GITHUB_BRANCH, f"origin/{GITHUB_BRANCH}"], token)
    os.makedirs(WORK_DIR, exist_ok=True)
else:   # yerel bilgisayarda: notebooks/ klasörünün bir üstü
    CODE_DIR = WORK_DIR = os.path.abspath("..")

sys.path.insert(0, CODE_DIR)
os.chdir(WORK_DIR)
import config
importlib.reload(config)
for d in [config.DATA_RAW, config.DATA_PROCESSED, config.TABLES_DIR, config.FIGURES_DIR, config.MODELS_DIR]:
    os.makedirs(d, exist_ok=True)
commit = subprocess.run(["git", "-C", CODE_DIR, "log", "-1", "--format=%h %cd", "--date=short"],
                        capture_output=True, text=True).stdout.strip()
print("Kod     :", CODE_DIR, "| dal:", GITHUB_BRANCH if IN_COLAB else "yerel", "| sürüm:", commit)
print("Sonuçlar:", WORK_DIR)

In [ ]:
import json, time, shutil, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import rasterio
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
T, F, M = config.TABLES_DIR, config.FIGURES_DIR, config.MODELS_DIR
CIKTI_TIF = "results/unet_tahmin"
for d in (f"{M}/unet", CIKTI_TIF):
    os.makedirs(d, exist_ok=True)

CIHAZ = "cuda" if torch.cuda.is_available() else "cpu"
if CIHAZ == "cpu":
    print("UYARI: GPU yok. Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU. CPU'da eğitim çok yavaş olur.")
else:
    print("GPU:", torch.cuda.get_device_name(0))

def tohum(s=config.RANDOM_STATE):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
tohum()

def savefig(name):
    plt.tight_layout()
    plt.savefig(f"{F}/{name}.png", dpi=200, bbox_inches="tight")
    plt.show()

LABELS = sorted(config.CLASSES)
SINIF_ADI = [config.CLASSES[k] for k in LABELS]
SINIF_RENK = [config.CLASS_COLORS[k] for k in LABELS]
NS, YOK, P = len(LABELS), config.S2_ETIKET_YOK, config.UNET_PATCH
YIL = json.load(open(f"{config.DATA_PROCESSED}/s2_yillar.json", encoding="utf-8"))
ILK, ETIKET, SON = YIL["ilk_yil"], YIL["etiket_yili"], YIL["son_yil"]
KUTULAR = YIL["kutular"]
IL_OF = {k: k.split("_")[0] for k in KUTULAR}
print("Kutular:", len(KUTULAR), "| yıllar:", ILK, ETIKET, SON)

## 1. Veriyi yükle ve şeritlere böl
Drive'dan okumak yavaş olduğu için GeoTIFF'ler önce Colab'ın yerel diskine kopyalanır.

**Bölme (mekânsal sızıntıya karşı):** Her kutu kuzeyden güneye üç şeride ayrılır: üst %70 **eğitim**, sonraki %15 **doğrulama**, alt %15 **test**.
Şeritler arasında 32 piksel (320 m) **tampon** bırakılır ve tampon değerlendirmeye girmez. Rastgele piksel bölmesi yerine bu yapılır, çünkü yan yana pikseller
birbirine çok benzer (03'teki mekânsal blok tartışması).

**Normalleştirme:** Her bant, eğitim şeritlerindeki ortalama ve standart sapmayla ölçeklenir (yalnızca eğitim verisinden: sızıntı yok).
Verisiz pikseller (bütün bantlar 0) etiketsiz sayılır.

In [ ]:
YEREL = "/content/s2" if os.path.exists("/content") else f"{config.DATA_RAW}/s2"
if YEREL != f"{config.DATA_RAW}/s2" and not os.path.exists(f"{YEREL}/.tamam"):
    shutil.copytree(f"{config.DATA_RAW}/s2", YEREL, dirs_exist_ok=True)
    open(f"{YEREL}/.tamam", "w").close()

def oku_goruntu(kutu, yil):
    with rasterio.open(f"{YEREL}/s2_{kutu}_{yil}.tif") as r:
        return r.read().astype(np.float32), r.profile

def oku_etiket(kutu):
    with rasterio.open(f"{YEREL}/s2_{kutu}_etiket.tif") as r:
        return r.read(1)

TAMPON = 32
def seritler(h):
    s1, s2 = int(config.UNET_BOLME[0] * h), int(sum(config.UNET_BOLME[:2]) * h)
    return {"egitim": (0, s1 - TAMPON), "dogrulama": (s1, s2 - TAMPON), "test": (s2, h)}

X, Y, PROFIL = {}, {}, {}
for kutu in KUTULAR:
    x, prof = oku_goruntu(kutu, ETIKET)
    y = oku_etiket(kutu).copy()
    y[(x == config.S2_NODATA).all(axis=0)] = YOK                   # verisiz piksel → etiketsiz
    X[kutu], Y[kutu], PROFIL[kutu] = x, y, prof
H, W = Y[KUTULAR[0]].shape
SERIT = seritler(H)
print("boyut:", H, "x", W, "| şeritler (satır):", SERIT)

# Bant ortalama/std: yalnızca eğitim şeritlerinden
toplam = np.zeros(len(config.S2_BANDS)); kare = np.zeros(len(config.S2_BANDS)); n = 0
for kutu in KUTULAR:
    a, b = SERIT["egitim"]
    x = X[kutu][:, a:b].reshape(len(config.S2_BANDS), -1)
    gecerli = Y[kutu][a:b].ravel() != YOK
    x = x[:, gecerli]
    toplam += x.sum(1); kare += (x.astype(np.float64) ** 2).sum(1); n += x.shape[1]
ORT = (toplam / n).astype(np.float32)
STD = np.sqrt(kare / n - ORT.astype(np.float64) ** 2).astype(np.float32)
json.dump({"ortalama": ORT.tolist(), "std": STD.tolist(), "bantlar": config.S2_BANDS},
          open(f"{M}/unet/normallestirme.json", "w"), indent=1)

def normallestir(x):
    return (x - ORT[:, None, None]) / STD[:, None, None]

for kutu in KUTULAR:
    X[kutu] = normallestir(X[kutu])

# Sınıf frekansı (eğitim) → kayıp ağırlıkları (az örnekli sınıf daha ağır, karekök ölçekli)
say = np.zeros(NS)
for kutu in KUTULAR:
    a, b = SERIT["egitim"]
    say += np.bincount(Y[kutu][a:b].ravel(), minlength=256)[:NS]
AGIRLIK = torch.tensor((say.sum() / (NS * np.maximum(say, 1))) ** 0.5, dtype=torch.float32)
dagilim = pd.DataFrame({"sinif": SINIF_ADI, "egitim_piksel": say.astype(int), "pay_yuzde": (100 * say / say.sum()).round(2),
                        "kayip_agirligi": AGIRLIK.numpy().round(3)})
dagilim.to_csv(f"{T}/unet_sinif_dagilimi.csv", index=False, encoding="utf-8-sig")
dagilim

## 2. Eğitim parçaları ve model
- **Eğitim:** Her epoch'ta eğitim şeritlerinden rastgele `256 × 256` parçalar alınır (`UNET_ORNEK_EPOCH` adet) ve rastgele döndürülür/yansıtılır
  (veri artırma). Etiketinin yarısından fazlası boş olan parça atlanır.
- **Model:** `segmentation_models_pytorch` kütüphanesinin U-Net'i; ResNet-34 kodlayıcı ImageNet ağırlıklarıyla başlar, ilk katman 6 banda uyarlanır (transfer öğrenme).
- **Kayıp:** Ağırlıklı çapraz entropi + Dice. Dice, az piksel kaplayan sınıfları (su, yapılaşma) öne çıkarır.
- **Tahmin:** Bütün kutu, %25 örtüşen 256'lık pencerelerle taranır ve örtüşen yerlerde olasılıklar ortalanır (pencere kenarı izi kalmaz).

In [ ]:
class RastgeleParca(torch.utils.data.Dataset):
    def __init__(self, kutular, n, serit="egitim"):
        self.kutular, self.n, (self.a, self.b) = kutular, n, SERIT[serit]
    def __len__(self):
        return self.n
    def __getitem__(self, i):
        for _ in range(20):
            kutu = random.choice(self.kutular)
            r = random.randint(self.a, self.b - P); c = random.randint(0, W - P)
            y = Y[kutu][r:r + P, c:c + P]
            if (y != YOK).mean() > 0.5:
                break
        x = X[kutu][:, r:r + P, c:c + P]
        k = random.randint(0, 3)
        x, y = np.rot90(x, k, axes=(1, 2)), np.rot90(y, k)
        if random.random() < 0.5:
            x, y = x[:, :, ::-1], y[:, ::-1]
        return torch.from_numpy(x.copy()), torch.from_numpy(y.astype(np.int64).copy())

def yeni_model():
    return smp.Unet(config.UNET_ENCODER, encoder_weights="imagenet", in_channels=len(config.S2_BANDS), classes=NS).to(CIHAZ)

ce = nn.CrossEntropyLoss(weight=AGIRLIK.to(CIHAZ), ignore_index=YOK)
dice = smp.losses.DiceLoss("multiclass", ignore_index=YOK)
def kayip(cikti, y):
    return ce(cikti, y) + dice(cikti, y)

@torch.no_grad()
def tahmin_kutu(model, x, adim=192):
    """Bütün kutuyu örtüşen pencerelerle tarar; sınıf haritası (H, W) döndürür."""
    model.eval()
    _, h, w = x.shape
    olas = torch.zeros((NS, h, w), device=CIHAZ); sayac = torch.zeros((h, w), device=CIHAZ)
    rs = sorted(set(list(range(0, h - P, adim)) + [h - P])); cs = sorted(set(list(range(0, w - P, adim)) + [w - P]))
    xt = torch.from_numpy(x).to(CIHAZ)
    for r in rs:
        parcalar = torch.stack([xt[:, r:r + P, c:c + P] for c in cs])
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=CIHAZ == "cuda"):
            o = torch.softmax(model(parcalar).float(), 1)
        for j, c in enumerate(cs):
            olas[:, r:r + P, c:c + P] += o[j]; sayac[r:r + P, c:c + P] += 1
    return (olas / sayac).argmax(0).cpu().numpy().astype(np.uint8)

def karisiklik(gercek, tahmin):
    m = gercek != YOK
    return np.bincount(NS * gercek[m].astype(int) + tahmin[m].astype(int), minlength=NS * NS).reshape(NS, NS)

def metrikler(cm):
    """Macro ortalamalar yalnızca gerçekte (etikette) bulunan sınıflar üzerinden alınır.
    Bir bölgede hiç olmayan sınıfı 0 puanla ortalamaya katmak, sınıf çeşitliliği az olan illerde
    (ör. %90'ı tarım olan Şanlıurfa) macro-F1'i yapay olarak düşürür. Olmayan sınıfın değeri NaN'dır."""
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    var = cm.sum(1) > 0                                            # etikette bulunan sınıflar
    prec = tp / np.maximum(tp + fp, 1); rec = tp / np.maximum(tp + fn, 1)
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-12); iou = tp / np.maximum(tp + fp + fn, 1)
    f1, iou, prec, rec = (np.where(var, a, np.nan) for a in (f1, iou, prec, rec))
    return {"accuracy": tp.sum() / cm.sum(), "precision_macro": np.nanmean(prec), "recall_macro": np.nanmean(rec),
            "f1_macro": np.nanmean(f1), "iou_macro": np.nanmean(iou), "sinif_sayisi": int(var.sum()),
            "f1_sinif": f1, "iou_sinif": iou}

def serit_cm(model, kutular, serit, tahminler=None):
    a, b = SERIT[serit]
    cm = np.zeros((NS, NS), dtype=np.int64)
    for kutu in kutular:
        t = tahminler[kutu] if tahminler is not None else tahmin_kutu(model, X[kutu])
        cm += karisiklik(Y[kutu][a:b], t[a:b])
    return cm

## 3. Eğitim
Her epoch'ta doğrulama şeritlerinde **mIoU** (ortalama IoU) ölçülür. En iyi model `results/models/unet/en_iyi.pt` dosyasına, her epoch'un durumu
`son_durum.pt` dosyasına kaydedilir. Oturum koparsa bu hücre kaldığı epoch'tan devam eder. `UNET_PATIENCE` epoch boyunca iyileşme olmazsa eğitim durur.
Baştan eğitmek için `BASTAN = True` yapın.

In [ ]:
BASTAN = False
# True: modeller Drive'daki kayıtlardan yüklenir (YENİDEN EĞİTİLMEZ), bütün metrik ve tablolar yeniden hesaplanır
# (il dışı test tablosu dahil). Metrik tanımı değiştiğinde ya da tabloları yenilemek için kullanın.
YENIDEN_DEGERLENDIR = True

def egit(kutular, klasor, epochs, patience=None, dogrulama_kutulari=None):
    """U-Net eğitir; her epoch sonunda Drive'a kaydeder, yarıda kalırsa devam eder. En iyi modeli döndürür."""
    os.makedirs(klasor, exist_ok=True)
    durum_yolu, en_iyi_yolu = f"{klasor}/son_durum.pt", f"{klasor}/en_iyi.pt"
    model = yeni_model()
    opt = torch.optim.AdamW(model.parameters(), lr=config.UNET_LR, weight_decay=1e-4)
    zam = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    olcek = torch.amp.GradScaler("cuda", enabled=CIHAZ == "cuda")
    gecmis, bas, en_iyi, sabir = [], 0, -1.0, 0
    if os.path.exists(durum_yolu) and not BASTAN:
        d = torch.load(durum_yolu, map_location=CIHAZ, weights_only=False)
        model.load_state_dict(d["model"]); opt.load_state_dict(d["opt"]); zam.load_state_dict(d["zam"])
        olcek.load_state_dict(d["olcek"]); gecmis, bas, en_iyi, sabir = d["gecmis"], d["epoch"] + 1, d["en_iyi"], d["sabir"]
        print(f"devam ediliyor: epoch {bas} | en iyi doğrulama mIoU {en_iyi:.4f}")
    yukleyici = torch.utils.data.DataLoader(RastgeleParca(kutular, config.UNET_ORNEK_EPOCH), batch_size=config.UNET_BATCH,
                                            shuffle=True, num_workers=2, drop_last=True, worker_init_fn=lambda i: tohum(config.RANDOM_STATE + i + bas))
    for ep in range(bas, epochs):
        if patience is not None and sabir >= patience:
            print("erken durdurma"); break
        model.train(); t0, toplam = time.time(), 0.0
        for xb, yb in yukleyici:
            xb, yb = xb.to(CIHAZ, non_blocking=True), yb.to(CIHAZ, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=CIHAZ == "cuda"):
                l = kayip(model(xb), yb)
            olcek.scale(l).backward(); olcek.step(opt); olcek.update()
            toplam += l.item()
        zam.step()
        satir = {"epoch": ep, "kayip": toplam / len(yukleyici), "sure_sn": round(time.time() - t0)}
        if dogrulama_kutulari:
            mt = metrikler(serit_cm(model, dogrulama_kutulari, "dogrulama"))
            satir.update(dogrulama_miou=mt["iou_macro"], dogrulama_f1=mt["f1_macro"])
            if mt["iou_macro"] > en_iyi:
                en_iyi, sabir = mt["iou_macro"], 0
                torch.save(model.state_dict(), en_iyi_yolu)
            else:
                sabir += 1
        else:
            torch.save(model.state_dict(), en_iyi_yolu)
        gecmis.append(satir)
        print(satir)
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "zam": zam.state_dict(), "olcek": olcek.state_dict(),
                    "gecmis": gecmis, "epoch": ep, "en_iyi": en_iyi, "sabir": sabir}, durum_yolu)
    model.load_state_dict(torch.load(en_iyi_yolu, map_location=CIHAZ))
    return model, pd.DataFrame(gecmis)

model, gecmis = egit(KUTULAR, f"{M}/unet/ana", config.UNET_EPOCHS, config.UNET_PATIENCE, dogrulama_kutulari=KUTULAR)
gecmis.to_csv(f"{T}/unet_egitim_gecmisi.csv", index=False, encoding="utf-8-sig")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(gecmis["epoch"], gecmis["kayip"]); ax[0].set_title("Eğitim kaybı"); ax[0].set_xlabel("epoch")
ax[1].plot(gecmis["epoch"], gecmis["dogrulama_miou"], label="mIoU"); ax[1].plot(gecmis["epoch"], gecmis["dogrulama_f1"], label="macro-F1")
ax[1].set_title("Doğrulama şeritleri"); ax[1].set_xlabel("epoch"); ax[1].legend()
savefig("unet_egitim_egrileri")

## 4. Test sonuçları
Bütün kutular bir kez tahmin edilir; metrikler yalnızca **test şeritlerinde** hesaplanır (model bu şeritleri hiç görmedi).
Accuracy, Precision, Recall, F1 (macro) ve **IoU**; sınıf bazında IoU ve karışıklık matrisi. Ayrıca il bazında macro-F1.

**Macro ortalama**, test bölgesinde gerçekten bulunan sınıflar üzerinden alınır (`sinif_sayisi` sütunu). Bir ilde hiç olmayan sınıfı 0 puanla saymak, sınıf çeşitliliği az olan illerde sonucu yapay olarak düşürürdü.

In [ ]:
TAHMIN = {kutu: tahmin_kutu(model, X[kutu]) for kutu in KUTULAR}
cm_test = serit_cm(model, KUTULAR, "test", TAHMIN)
mt = metrikler(cm_test)
unet_tab = pd.DataFrame([{k: round(float(v), 4) for k, v in mt.items() if not k.endswith("_sinif")}], index=["U-Net (Sentinel-2)"])
unet_tab.to_csv(f"{T}/unet_metrikleri.csv", encoding="utf-8-sig")
sinif_tab = pd.DataFrame({"F1": mt["f1_sinif"], "IoU": mt["iou_sinif"]}, index=SINIF_ADI).round(4)
sinif_tab.to_csv(f"{T}/unet_sinif_bazinda.csv", encoding="utf-8-sig")

il_satir = []
for il in config.CITIES:
    cm = serit_cm(model, [k for k in KUTULAR if IL_OF[k] == il], "test", TAHMIN)
    m_ = metrikler(cm)
    il_satir.append({"il": il, "f1_macro": m_["f1_macro"], "iou_macro": m_["iou_macro"], "accuracy": m_["accuracy"],
                     "sinif_sayisi": m_["sinif_sayisi"], "piksel": int(cm.sum())})
il_tab = pd.DataFrame(il_satir).round(4)
il_tab.to_csv(f"{T}/unet_il_bazinda.csv", index=False, encoding="utf-8-sig")
display(unet_tab); display(sinif_tab); display(il_tab)

fig, ax = plt.subplots(figsize=(7, 6))
yuzde = 100 * cm_test / np.maximum(cm_test.sum(1, keepdims=True), 1)
sns.heatmap(yuzde, annot=True, fmt=".0f", cmap="Blues", vmin=0, vmax=100, cbar=False, xticklabels=SINIF_ADI, yticklabels=SINIF_ADI, ax=ax)
ax.set_xlabel("Tahmin"); ax.set_ylabel("Gerçek (WorldCover)"); ax.set_title("U-Net — karışıklık matrisi (test şeritleri, satır %)")
savefig("unet_karisiklik_matrisi")

## 5. Önce RF, sonra U-Net: aynı test pikselleri üzerinde karşılaştırma
GeoClimate dokümanı "önce Random Forest ile temel model, sonra U-Net" der. Adil karşılaştırma için RF **aynı Sentinel-2 verisi ve aynı bölmeyle** eğitilir:
eğitim şeritlerinden rastgele `UNET_RF_PIKSEL` piksel; özellikler 6 bant + NDVI, NDBI, MNDWI (03'teki gibi). RF pikseli **tek başına** sınıflandırır; U-Net
komşularına da bakar. Fark, **mekânsal bağlamın** katkısını gösterir. Değerlendirme test şeritlerinden rastgele 500 000 pikselde, iki model için aynı piksellerde yapılır.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

def ozellik(x_norm):
    x = x_norm * STD[:, None] + ORT[:, None]                      # ham yansıma (indeksler için)
    b, g, r, nir, sw1, sw2 = x
    e = 1e-6
    return np.vstack([x, (nir - r) / (nir + r + e), (sw1 - nir) / (sw1 + nir + e), (g - sw1) / (g + sw1 + e)]).T

def piksel_ornekle(kutular, serit, n, rng):
    a, b = SERIT[serit]
    xs, ys, idx = [], [], []
    for kutu in kutular:
        y = Y[kutu][a:b].ravel(); gecerli = np.flatnonzero(y != YOK)
        sec = rng.choice(gecerli, size=min(n // len(kutular), len(gecerli)), replace=False)
        xs.append(ozellik(X[kutu][:, a:b].reshape(len(config.S2_BANDS), -1)[:, sec])); ys.append(y[sec])
        idx.append((kutu, sec))
    return np.vstack(xs), np.concatenate(ys), idx

rng = np.random.default_rng(config.RANDOM_STATE)
Xtr, ytr, _ = piksel_ornekle(KUTULAR, "egitim", config.UNET_RF_PIKSEL, rng)
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=config.RANDOM_STATE).fit(Xtr, ytr)
Xte, yte, idx = piksel_ornekle(KUTULAR, "test", 500_000, rng)
p_rf = rf.predict(Xte)
a, b = SERIT["test"]
p_unet = np.concatenate([TAHMIN[kutu][a:b].ravel()[sec] for kutu, sec in idx])
kars = []
for ad, p in [("Random Forest (piksel, Sentinel-2)", p_rf), ("U-Net (Sentinel-2)", p_unet)]:
    m_ = metrikler(karisiklik(yte, p))
    kars.append({"model": ad, **{k: round(float(v), 4) for k, v in m_.items() if not k.endswith("_sinif")},
                 **{f"IoU_{s}": round(float(v), 4) for s, v in zip(SINIF_ADI, m_["iou_sinif"])}})
kars = pd.DataFrame(kars).set_index("model")
kars.to_csv(f"{T}/unet_vs_rf.csv", encoding="utf-8-sig")
display(kars)
kars[["accuracy", "f1_macro", "iou_macro"]].T.plot(kind="bar", figsize=(8, 4), color=["#bababa", "#1b7837"])
plt.ylim(0, 1); plt.title("Aynı test piksellerinde RF ve U-Net"); plt.xticks(rotation=0)
savefig("unet_vs_rf")

## 6. H6 — il dışı test (Leave-One-City-Out)
Bir ilin **iki kutusu da** eğitimden çıkarılır; model kalan 6 ilin eğitim şeritleriyle eğitilip çıkarılan ilin test şeritlerinde ölçülür. Karşılaştırma için
ana modelin (o ili görmüş) aynı test şeritlerindeki sonucu yan yana verilir. Süre kısalsın diye daha az epoch kullanılır (`UNET_LOCO_EPOCHS`) ve erken durdurma yoktur.
Oturum koparsa tekrar çalıştırın: biten iller atlanır (`YENIDEN_DEGERLENDIR = True` iken tablo baştan hesaplanır, ama eğitimi biten iller kayıttan yüklenir, yeniden eğitilmez). Atlamak için `LOCO_CALISTIR = False`.

In [ ]:
LOCO_CALISTIR = True
loco_csv = f"{T}/unet_loco.csv"
loco = pd.read_csv(loco_csv) if os.path.exists(loco_csv) and not YENIDEN_DEGERLENDIR else pd.DataFrame(columns=["il", "f1_ayni", "f1_baska", "iou_ayni", "iou_baska", "dusus_f1"])
if LOCO_CALISTIR:
    for il in config.CITIES:
        if il in set(loco["il"]):
            print(il, "zaten var, atlandı"); continue
        disari = [k for k in KUTULAR if IL_OF[k] == il]
        m_il, _ = egit([k for k in KUTULAR if IL_OF[k] != il], f"{M}/unet/loco_{il}", config.UNET_LOCO_EPOCHS)
        baska = metrikler(serit_cm(m_il, disari, "test"))
        ayni = metrikler(serit_cm(model, disari, "test", TAHMIN))
        loco = pd.concat([loco, pd.DataFrame([{"il": il, "f1_ayni": ayni["f1_macro"], "f1_baska": baska["f1_macro"],
                                               "iou_ayni": ayni["iou_macro"], "iou_baska": baska["iou_macro"],
                                               "dusus_f1": ayni["f1_macro"] - baska["f1_macro"]}])], ignore_index=True).round(4)
        loco.to_csv(loco_csv, index=False, encoding="utf-8-sig")
        print(loco.iloc[-1].to_dict())
        del m_il; torch.cuda.empty_cache()
display(loco)
if len(loco):
    loco.set_index("il")[["f1_ayni", "f1_baska"]].plot(kind="bar", figsize=(10, 4), color=["#4575b4", "#d73027"])
    plt.ylabel("macro-F1 (test şeritleri)"); plt.title(f"U-Net: aynı il vs hiç görülmemiş il · ortalama düşüş {loco['dusus_f1'].mean():.3f}")
    plt.xticks(rotation=0); savefig("unet_loco")

## 7. Değişim: ilk yıl → son yıl
Ana model, ilk yıl ve son yıl görüntülerine uygulanır (aynı normalleştirmeyle). İki yılda da verisi olan pikseller karşılaştırılır:
**geçiş matrisi** (km²; 1 piksel = 100 m² = 0,0001 km²) ve **değişim haritası** (değişen pikseller yeni sınıfın renginde).
Tahminler GeoTIFF olarak da kaydedilir (`results/unet_tahmin/`); QGIS'te açılabilir.

**Sınırlılık:** Etiket yalnızca 2021 için var; ilk yıl ve 2025 haritalarının doğruluğu ölçülemiyor. Yıllar arası atmosfer ve fenoloji farkı sahte değişim
üretebilir; küçük değişimler temkinli yorumlanmalı (03–04'teki Landsat analiziyle aynı sınırlılık).

In [ ]:
from matplotlib.colors import ListedColormap
renk = ListedColormap(SINIF_RENK)
gecis, alan = [], []
for kutu in KUTULAR:
    yil_harita = {}
    for yil in (ILK, SON):
        x, prof = oku_goruntu(kutu, yil)
        verisiz = (x == config.S2_NODATA).all(axis=0)
        t = tahmin_kutu(model, normallestir(x))
        t[verisiz] = YOK
        yil_harita[yil] = t
        p = PROFIL[kutu].copy(); p.update(count=1, dtype="uint8", nodata=YOK, compress="deflate")
        with rasterio.open(f"{CIKTI_TIF}/{kutu}_{yil}.tif", "w", **p) as dst:
            dst.write(t, 1)
        say = np.bincount(t.ravel(), minlength=256)[:NS]
        alan += [{"kutu": kutu, "il": IL_OF[kutu], "yil": yil, "sinif": s, "km2": round(n * 1e-4, 3)} for s, n in zip(SINIF_ADI, say)]
    t0, t1 = yil_harita[ILK], yil_harita[SON]
    m = (t0 != YOK) & (t1 != YOK)
    ct = np.bincount(NS * t0[m].astype(int) + t1[m], minlength=NS * NS).reshape(NS, NS)
    for i in range(NS):
        for j in range(NS):
            gecis.append({"kutu": kutu, "il": IL_OF[kutu], f"sinif_{ILK}": SINIF_ADI[i], f"sinif_{SON}": SINIF_ADI[j], "km2": round(ct[i, j] * 1e-4, 3)})
    degisim = np.where(m & (t0 != t1), t1, np.nan).astype(float)
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    for a_, (baslik, img) in zip(ax, [(f"{ILK}", np.where(t0 == YOK, np.nan, t0)), (f"{SON}", np.where(t1 == YOK, np.nan, t1)), ("değişim", degisim)]):
        a_.imshow(np.full_like(degisim, 0.5) if baslik == "değişim" else img, cmap="Greys" if baslik == "değişim" else renk,
                  vmin=0 if baslik == "değişim" else -0.5, vmax=1 if baslik == "değişim" else 5.5, interpolation="nearest")
        if baslik == "değişim":
            a_.imshow(degisim, cmap=renk, vmin=-0.5, vmax=5.5, interpolation="nearest")
        a_.set_title(f"{kutu} · {baslik}"); a_.axis("off")
    from matplotlib.patches import Patch
    fig.legend(handles=[Patch(color=c, label=s) for s, c in zip(SINIF_ADI, SINIF_RENK)], loc="lower center", ncol=6)
    savefig(f"unet_degisim_{kutu}")
gecis = pd.DataFrame(gecis); alan = pd.DataFrame(alan)
gecis.to_csv(f"{T}/unet_degisim_matrisi.csv", index=False, encoding="utf-8-sig")
alan.to_csv(f"{T}/unet_sinif_alanlari.csv", index=False, encoding="utf-8-sig")
ozet = alan.pivot_table(index=["il", "sinif"], columns="yil", values="km2", aggfunc="sum")
ozet["degisim_km2"] = ozet[SON] - ozet[ILK]
ozet.round(2).to_csv(f"{T}/unet_degisim_ozet.csv", encoding="utf-8-sig")
ozet.round(2)

## Faz bitti mi?
- [ ] `unet_metrikleri.csv`, `unet_sinif_bazinda.csv`, `unet_il_bazinda.csv`, `unet_karisiklik_matrisi.png`, `unet_egitim_egrileri.png`
- [ ] `unet_vs_rf.csv`, `unet_vs_rf.png` (önce RF, sonra U-Net)
- [ ] `unet_loco.csv`, `unet_loco.png` (H6)
- [ ] `unet_degisim_matrisi.csv`, `unet_degisim_ozet.csv`, `unet_degisim_<kutu>.png`, `results/unet_tahmin/*.tif`
- [ ] Model: `results/models/unet/ana/en_iyi.pt` ve `normallestirme.json`

## Bu fazda ne yaptık, neden yaptık
- **U-Net** pikselin komşularına bakan bir segmentasyon ağıdır; tarla sınırı, kent dokusu, kıyı çizgisi gibi **şekil bilgisini** kullanır. 03'teki modeller
  her pikseli tek başına sınıflandırıyordu.
- **Transfer öğrenme:** ImageNet'te eğitilmiş ResNet-34 kodlayıcıyla başladık; az veriyle daha hızlı ve daha iyi öğrenir.
- **Şerit bölmesi ve tampon:** Test şeritleri eğitim şeritlerinden 320 m uzak; yan yana piksellerin sızıntısını önler.
- **Önce RF, sonra U-Net:** Aynı veri ve aynı test piksellerinde karşılaştırma, mekânsal bağlamın katkısını ölçer.
- **LOCO (H6):** Modelin hiç görmediği ilde başarısı; 06'daki Landsat sonuçlarıyla karşılaştırılabilir.
- **Değişim:** İlk yıl → 2025, 10 m çözünürlükte; doğruluk 2021 dışında ölçülemediği için yön, büyüklükten daha güvenilirdir.